### 1. Basic Tasks

In [0]:
-- 1. Use MERGE INTO to upsert a batch of 'changed customer' records into a silver customers table.

In [0]:
create table if not exists cyntexa_dev.bronze.customers (customer_id int, name string, email string, city string, state string, updated_at timestamp default current_timestamp())
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported')

In [0]:
insert into cyntexa_dev.bronze.customers (customer_id, name, email, city, state) values 
(101, 'Aarav Sharma', 'aarav@gmail.com', 'Jaipur', 'Rajasthan'),
(102, 'Ananya Gupta', 'ananya@gmail.com', 'Delhi', 'Delhi'),
(103, 'Rohan Mehta', 'rohan@gmail.com', 'Mumbai', 'Maharashtra'),
(104, 'Priya Singh', 'priya@gmail.com', 'Pune', 'Maharashtra'),
(105, 'Aditya Verma', 'aditya@gmail.com', 'Bangalore', 'Karnataka'),
(106, 'Neha Jain', 'neha@gmail.com', 'Ahmedabad', 'Gujarat'),
(107, 'Raj Malhotra', 'raj@gmail.com', 'Chandigarh', 'Chandigarh'),
(108, 'Isha Patel', 'isha@gmail.com', 'Surat', 'Gujarat'),
(109, 'Kunal Agarwal', 'kunal@gmail.com', 'Kolkata', 'West Bengal'),
(110, 'Simran Kaur', 'simran@gmail.com', 'Ludhiana', 'Punjab');

In [0]:
create table cyntexa_dev.bronze.customers_scd as select * except (updated_at) from cyntexa_dev.bronze.customers

In [0]:
merge into cyntexa_dev.bronze.customers_scd t 
using (
    select * from cyntexa_dev.bronze.customers 
    qualify rank() over (partition by customer_id order by updated_at desc) = 1
) s
on t.customer_id = s.customer_id
when matched then 
update set t.name = s.name,t.email = s.email , t.city = s.city, t.state = s.state
when not matched then 
insert(customer_id, name, email, city, state) values (s.customer_id, s.name, s.email, s.city, s.state)

In [0]:
select * from cyntexa_dev.bronze.customers_scd where customer_id = 101

In [0]:
insert into cyntexa_dev.bronze.customers (customer_id, name, email, city, state) values 
(101, 'Aarav Sharma', 'aarav@gmail.com', 'Mumbai',"Maharashtra")

Now run MERGE INTO script again to update in scd table

In [0]:
select * from cyntexa_dev.bronze.customers_scd where customer_id = 101

Now the customers_scd table contains updated value for customer_id 101

In [0]:
-- 2. Grant SELECT on a table to one group and a masked/limited view to another group using Unity Catalog permissions.

Created 2 groups Data Engineers and Data Analyst in Identity and Access management now grantig following permisions
- Data Engineers group: Can see all tables/ columns wihtout masking
- Data Analyst group: can see only silver/masked tables aor views 

In [0]:
grant select 
on table cyntexa_dev.bronze.customers
to `data_engineers` 

In [0]:
create function cyntexa_dev.bronze.mask_email(email string)
returns string
return concat(
    substring(email,1,2),
    '****',
    substring(email,instr(email,'@'),length(email))
)

In [0]:
create view masked_customer as
select customer_id, name, cyntexa_dev.bronze.mask_email(email) as email, city,
state from cyntexa_dev.bronze.customers

In [0]:
grant select 
on view masked_customer
to `data_analysts`

In [0]:
-- 3. Look up the DBU consumption for a compute resource you've been using and explain, in plain terms, what a DBU is billing for.

DBU stands for Databricks Units and is a unit used to measure Databricks compute consumption. The number of DBUs consumed depends on the type and configuration of the compute resource and how long it runs.

DBUs measure the amount of Databricks compute capaciry used and calculates compute cost.

### 2. Intermediate Tasks

In [0]:
-- 4. Build a full SCD Type 2 table: implement the MERGE that closes out old records (setting end_date and is_current) and inserts new versions when a tracked column changes.

In [0]:
create table cyntexa_dev.silver.products (
    product_id int, 
    name string,
    category string,
    price double
)

In [0]:
INSERT INTO cyntexa_dev.silver.products
VALUES
(1, 'Product1', 'Category1', 100.00),
(2, 'Product2', 'Category2', 150.00),
(3, 'Product3', 'Category1', 200.00),
(4, 'Product4', 'Category2', 250.00),
(5, 'Product5', 'Category1', 300.00),
(6, 'Product6', 'Category2', 350.00),
(7, 'Product7', 'Category1', 400.00),
(8, 'Product8', 'Category2', 450.00),
(9, 'Product9', 'Category1', 500.00),
(10, 'Product10', 'Category2', 550.00);

In [0]:
create table cyntexa_dev.silver.products_scd2 as 
select *,
cast(null as date) as effective_date,
cast(null as date) as end_date,
cast(null as int) as version,
cast(null  as boolean) as is_current
from cyntexa_dev.silver.products
where 1 = 0

In [0]:
merge into cyntexa_dev.silver.products_scd2 t
using cyntexa_dev.silver.products s 
on t.product_id = s.product_id and t.is_current = true
when matched and (not(t.name <=> s.name) or
not(t.category <=> s.category) or
not(t.price <=> s.price))
then update set
t.end_date = current_date(),
t.is_current = false
when not matched then
insert (product_id, name, category, price, effective_date, end_date, version, is_current)
values (s.product_id, s.name, s.category, s.price, current_date(), null, 1, true);

insert into cyntexa_dev.silver.products_scd2
select
s.product_id,
s.name,
s.category,
s.price,
current_date() as effective_date,
null as end_date,
coalesce(max(t.version),0) + 1 as version,
true as is_current
from cyntexa_dev.silver.products s
inner join cyntexa_dev.silver.products_scd2 t 
on t.product_id = s.product_id and t.end_date = current_date()
where not(t.name <=> s.name) or
not(t.category <=> s.category) or
not(t.price <=> s.price)
group by s.product_id,
s.name,
s.category,
s.price

In [0]:
select * from cyntexa_dev.silver.products_scd2

In [0]:
insert into cyntexa_dev.silver.products values(1, 'Product1', 'Category5', 110.00)

In [0]:
-- 5. Query the SCD Type 2 table to answer a point-in-time question, e.g. 'what was this customer's address as of March 1st?'
select * from cyntexa_dev.silver.products_scd2
where product_id = 1
and effective_date <= '2026-09-10' and end_date is null

In [0]:
-- 6. Compare the estimated DBU cost of running a job on all-purpose vs. job compute, and recommend which Cyntexa should use for its nightly pipeline.

All purpose compute vs Job compute
- All purpose compute: designed for interactive work such as developting, testing and running notebooks manually 
- Job compute: Designed specifically automated jobs and scheduled pipelies, where compute is started for the job and terminated with completion

All purpose Computes costs more than Jpb compute for the same workload because it is intended for interactive usage.

For Cyntexa's nightly pipeline, choose Job compute because the pipeline is an automated, scheduled workload and does not require and interactive cluster to remain available. This provide bettere cost effeciency and avoids paying for compute when pipeline is not running

###3. Advanced Tasks

In [0]:
-- 7. Design a governance model for Cyntexa: which columns across which tables are sensitive (PII), which Unity Catalog groups should have access, and how you'd audit access after the fact.

Cyntexa should classify **email, phone number, and address** as sensitive customer PII.

**Access Model:**

* **Data Engineers:** Full access to Silver customer tables as required for data processing and pipeline development.
* **Analysts:** Access limited to approved Gold tables and views, with PII masked where applicable.
* **Managers:** Access to aggregated business information without exposure to raw customer PII.
* **Restricted PII Access:** Raw customer PII should only be accessible to specifically authorized users or groups.

**Audit:**

Unity Catalog audit logs and system tables should be used to track access to sensitive data, including **who accessed it, when it was accessed, and what operation was performed**.

**Principle:**

The governance model should follow the **principle of least privilege**, ensuring that each user receives only the level of data access necessary for their responsibilities.


In [0]:
-- 8. Extend the SCD Type 2 pattern to track changes across 3+ columns simultaneously, and handle the edge case of a customer record that hasn't changed since the last load (it should not create a false new version).

In [0]:
CREATE TABLE cyntexa_dev.silver.customer (
    customer_id INT,
    customer_name STRING,
    email STRING,
    city STRING,
    phone STRING,
    customer_type STRING,
    credit_limit DECIMAL(10,2),
    customer_status STRING
);

In [0]:
INSERT INTO cyntexa_dev.silver.customer
(
    customer_id,
    customer_name,
    email,
    city,
    phone,
    customer_type,
    credit_limit,
    customer_status
)
VALUES
(1, 'Aarav Sharma', 'aarav.sharma@gmail.com', 'Delhi', '9876543210', 'Premium', 100000.00, 'Active'),
(2, 'Priya Verma', 'priya.verma@gmail.com', 'Mumbai', '9876543211', 'Standard', 50000.00, 'Active'),
(3, 'Rahul Singh', 'rahul.singh@gmail.com', 'Bangalore', '9876543212', 'Premium', 120000.00, 'Active'),
(4, 'Sneha Gupta', 'sneha.gupta@gmail.com', 'Pune', '9876543213', 'Standard', 60000.00, 'Active'),
(5, 'Vikram Mehta', 'vikram.mehta@gmail.com', 'Chennai', '9876543214', 'Basic', 30000.00, 'Active'),
(6, 'Ananya Patel', 'ananya.patel@gmail.com', 'Ahmedabad', '9876543215', 'Premium', 150000.00, 'Active'),
(7, 'Rohan Kapoor', 'rohan.kapoor@gmail.com', 'Hyderabad', '9876543216', 'Standard', 70000.00, 'Active'),
(8, 'Neha Joshi', 'neha.joshi@gmail.com', 'Jaipur', '9876543217', 'Basic', 25000.00, 'Active'),
(9, 'Karan Malhotra', 'karan.malhotra@gmail.com', 'Kolkata', '9876543218', 'Premium', 110000.00, 'Active'),
(10, 'Pooja Iyer', 'pooja.iyer@gmail.com', 'Kochi', '9876543219', 'Standard', 55000.00, 'Active');

In [0]:
CREATE TABLE cyntexa_dev.silver.customer_scd2 AS

SELECT
    *,
    CAST(NULL AS DATE) AS effective_date,
    CAST(NULL AS DATE) AS end_date,
    CAST(NULL AS INT) AS version,
    CAST(NULL AS BOOLEAN) AS is_current

FROM cyntexa_dev.silver.customer

WHERE 1 = 0;

In [0]:
MERGE INTO cyntexa_dev.silver.customer_scd2 t

USING cyntexa_dev.silver.customer s

ON t.customer_id = s.customer_id
AND t.is_current = true

WHEN MATCHED AND
(
    NOT(t.customer_name <=> s.customer_name)
    OR NOT(t.email <=> s.email)
    OR NOT(t.city <=> s.city)
    OR NOT(t.phone <=> s.phone)
    OR NOT(t.customer_type <=> s.customer_type)
    OR NOT(t.credit_limit <=> s.credit_limit)
    OR NOT(t.customer_status <=> s.customer_status)
)

THEN UPDATE SET

    t.end_date = current_date(),

    t.is_current = false

WHEN NOT MATCHED THEN

INSERT
(
    customer_id,
    customer_name,
    email,
    city,
    phone,
    customer_type,
    credit_limit,
    customer_status,
    effective_date,
    end_date,
    version,
    is_current
)

VALUES
(
    s.customer_id,
    s.customer_name,
    s.email,
    s.city,
    s.phone,
    s.customer_type,
    s.credit_limit,
    s.customer_status,
    current_date(),
    NULL,
    1,
    true
);

INSERT INTO cyntexa_dev.silver.customer_scd2

SELECT

    s.customer_id,

    s.customer_name,

    s.email,

    s.city,

    s.phone,

    s.customer_type,

    s.credit_limit,
    s.customer_status,

    current_date() AS effective_date,

    NULL AS end_date,

    COALESCE(MAX(t.version), 0) + 1 AS version,

    true AS is_current

FROM cyntexa_dev.silver.customer s

INNER JOIN cyntexa_dev.silver.customer_scd2 t

ON t.customer_id = s.customer_id
AND t.end_date = current_date()

WHERE

    NOT(t.customer_name <=> s.customer_name)
    OR NOT(t.email <=> s.email)
    OR NOT(t.city <=> s.city)
    OR NOT(t.phone <=> s.phone)
    OR NOT(t.customer_type <=> s.customer_type)
    OR NOT(t.credit_limit <=> s.credit_limit)
    OR NOT(t.customer_status <=> s.customer_status)

GROUP BY

    s.customer_id,

    s.customer_name,

    s.email,

    s.city,

    s.phone,

    s.customer_type,

    s.credit_limit,
    
    s.customer_status;

In [0]:
select * from cyntexa_dev.silver.customer_scd2

In [0]:
UPDATE cyntexa_dev.silver.customer

SET
    city = 'Noida',
    phone = '9999999999',
    credit_limit = 130000.00

WHERE customer_id = 1;

In [0]:
select * from cyntexa_dev.silver.customer_scd2 where customer_id = 1

Changing 3+ columns does nor create fake versions for each column update, instead update everything in one version

In [0]:
insert into cyntexa_dev.silver.customer values(10,"Pooja Iyer","pooja.iyer@gmail.com","Kochi",9876543219,"Standard",55000.00,"Active")

In [0]:
select * from cyntexa_dev.silver.customer_scd2 where customer_id = 10

inserting duplicate record did not create fake duplicate versions 

In [0]:
-- 9. (Data Analyst) Using the SCD Type 2 history table, build a customer retention/churn-over-time report that depends on point-in-time correctness, and explain why a simple 'current state' table would give the wrong answer here.

SCD Type 2 maintains historical versions of customer records, allowing us to identify whether a customer was active during a particular reporting period.

A customer is considered active if their SCD2 record was valid during that period.

Using only a current-state table would not provide accurate historical results because it stores only the customer's most recent information. When attributes such as address or status are updated, the previous values are replaced, making it impossible to determine the customer's state at an earlier point in time.

Therefore, SCD Type 2 is essential for preserving historical customer states and performing accurate retention and churn analysis.
